# Численное моделирование распространения сейсмических волн в двумерной среде MILEN SEM 2D. Часть вторая.

## Глава I: Модель для геофизиков

### Научная задачаНаше исследование заключается в построении конечноэлементной и конечноразностной моделей с последующим совместным их решением.Мы уже построили конечноэлементную модель для программы CAE-FidesysТеперь нужно построить аналогичную конечно-разностную двумерную модель задачи в файле формата, совместимого с tesseral.

In [ ]:
import numpy as npimport segyiofrom pathlib import Path

#### Задача 1: Анализ формата данных Tesseral ####

In [ ]:
# Анализируем пример из Tesseralexample_vp_path = 'src/tesseral_example/Vp_model.sgy'print("="*60)print("АНАЛИЗ ФОРМАТА SEG-Y ИЗ TESSERAL")print("="*60)with segyio.open(example_vp_path, 'r', ignore_geometry=True) as f:    print(f"\nФайл: {example_vp_path}")    print(f"Количество трасс: {f.tracecount}")    print(f"Количество сэмплов на трассу: {len(f.trace[0])}")    print(f"Формат данных: {f.format}")        # Читаем заголовок первой трассы    print(f"\nЗаголовок первой трассы:")    header = f.header[0]    print(f"  Inline: {header[segyio.TraceField.INLINE_3D]}")    print(f"  Crossline: {header[segyio.TraceField.CROSSLINE_3D]}")    print(f"  CDP X: {header[segyio.TraceField.CDP_X]}")    print(f"  CDP Y: {header[segyio.TraceField.CDP_Y]}")    print(f"  Sample interval: {header[segyio.TraceField.TRACE_SAMPLE_INTERVAL]}")        # Читаем бинарный заголовок    print(f"\nБинарный заголовок:")    bin_header = f.bin    print(f"  Job ID: {bin_header[segyio.BinField.JobID]}")    print(f"  Samples: {bin_header[segyio.BinField.Samples]}")    print(f"  Interval: {bin_header[segyio.BinField.Interval]}")    print(f"  Format: {bin_header[segyio.BinField.Format]}")        # Читаем данные первой трассы    trace_0 = f.trace[0]    print(f"\nДанные первой трассы:")    print(f"  Минимум: {trace_0.min():.2f}")    print(f"  Максимум: {trace_0.max():.2f}")    print(f"  Среднее: {trace_0.mean():.2f}")    print(f"  Первые 5 значений: {trace_0[:5]}")# Анализируем все три файлаfor param_name in ['Vp', 'Vs', 'Density']:    param_path = f'src/tesseral_example/{param_name}_model.sgy'    with segyio.open(param_path, 'r', ignore_geometry=True) as f:        data = segyio.tools.collect(f.trace[:])        print(f"\n{param_name}:")        print(f"  Форма данных: {data.shape}")        print(f"  Диапазон: [{data.min():.2f}, {data.max():.2f}]")

#### Задача 2: Подготовка и выгрузка наших данных ####

In [ ]:
print("\n" + "="*60)print("ПОДГОТОВКА ДАННЫХ ДЛЯ TESSERAL")print("="*60)# Загружаем наши декартовы данныеdata_path = Path('data/dev_1_7_material_grids.npz')data = np.load(data_path)material_grid = data['material_grid']      # [2350, 550, 3]coords_grid = data['coords_grid']          # [2350, 550, 2]layer_indexes_grid = data['layer_indexes_grid']  # [2350, 550]print(f"\nЗагружены данные из {data_path}")print(f"Размер сетки материала: {material_grid.shape}")print(f"Размер сетки координат: {coords_grid.shape}")# Извлекаем свойстваE_grid = material_grid[:, :, 0]   # Модуль Юнга (Па)nu_grid = material_grid[:, :, 1]  # Коэффициент Пуассонаrho_grid = material_grid[:, :, 2] # Плотность (кг/м³)# Вычисляем скорости волнVp_grid = np.sqrt(E_grid / rho_grid * (1 - nu_grid) /                   ((1 + nu_grid) * (1 - 2*nu_grid)))Vs_grid = np.sqrt(E_grid / (2 * rho_grid * (1 + nu_grid)))# Плотность в г/см³ (стандарт для геофизики)density_grid = rho_grid / 1000.0print(f"\nДиапазоны параметров:")print(f"  Vp: {Vp_grid.min():.1f} - {Vp_grid.max():.1f} м/с")print(f"  Vs: {Vs_grid.min():.1f} - {Vs_grid.max():.1f} м/с")print(f"  Плотность: {density_grid.min():.3f} - {density_grid.max():.3f} г/см³")# Получаем параметры сеткиx_coords = coords_grid[:, 0, 0]  # Координаты Xy_coords = coords_grid[0, :, 1]  # Координаты Y (глубины)dx = x_coords[1] - x_coords[0] if len(x_coords) > 1 else 5.0dy = y_coords[1] - y_coords[0] if len(y_coords) > 1 else 5.0print(f"\nПараметры сетки:")print(f"  Nx = {len(x_coords)}, Ny = {len(y_coords)}")print(f"  dx = {dx:.1f} м, dy = {dy:.1f} м")print(f"  X: {x_coords[0]:.1f} - {x_coords[-1]:.1f} м")print(f"  Y: {y_coords[0]:.1f} - {y_coords[-1]:.1f} м")

#### Функция для записи данных в формат SEG-Y ####

In [ ]:
def write_segy_2d(filename, data, dx=5.0, dy=5.0, x_origin=0.0, y_origin=0.0):    """    Записывает 2D массив данных в формат SEG-Y.        Args:        filename: имя выходного файла        data: 2D массив данных [nx, ny]        dx: шаг по X (м)        dy: шаг по Y (м)          x_origin: начало координат по X (м)        y_origin: начало координат по Y (м)    """    nx, ny = data.shape        # Создаем спецификацию SEG-Y    spec = segyio.spec()    spec.format = 5  # IEEE float    spec.samples = range(ny)  # индексы сэмплов    spec.tracecount = nx  # количество трасс = количество точек по X        # Интервал в микросекундах (для совместимости с сейсмикой)    # Используем dy в метрах, умножаем на 1000 для перевода в мкс    spec.interval = int(dy * 1000)        with segyio.create(filename, spec) as f:        # Записываем бинарный заголовок        f.bin = {            segyio.BinField.JobID: 1,            segyio.BinField.Samples: ny,            segyio.BinField.Interval: spec.interval,            segyio.BinField.Format: 5  # IEEE float        }                # Записываем трассы        for i in range(nx):            # Заголовок трассы            f.header[i] = {                segyio.TraceField.TRACE_SEQUENCE_FILE: i + 1,                segyio.TraceField.TRACE_SEQUENCE_LINE: i + 1,                segyio.TraceField.INLINE_3D: i + 1,                segyio.TraceField.CROSSLINE_3D: 1,                segyio.TraceField.CDP_X: int(x_origin + i * dx),                segyio.TraceField.CDP_Y: int(y_origin),                segyio.TraceField.TRACE_SAMPLE_COUNT: ny,                segyio.TraceField.TRACE_SAMPLE_INTERVAL: spec.interval            }                        # Данные трассы (столбец по Y)            f.trace[i] = data[i, :].astype(np.float32)        print(f"  Записан файл: {filename}")    print(f"    Размер: {nx} × {ny}")    print(f"    Диапазон: [{data.min():.2f}, {data.max():.2f}]")

#### Запись данных в файлы SEG-Y ####

In [ ]:
print("\n" + "="*60)print("ЗАПИСЬ ДАННЫХ В ФОРМАТ SEG-Y")print("="*60 + "\n")output_dir = Path('data')# Записываем Vpvp_output = output_dir / 'dev_2_1_Vp_model.sgy'write_segy_2d(str(vp_output), Vp_grid, dx=dx, dy=dy)# Записываем Vsvs_output = output_dir / 'dev_2_1_Vs_model.sgy'write_segy_2d(str(vs_output), Vs_grid, dx=dx, dy=dy)# Записываем плотностьdensity_output = output_dir / 'dev_2_1_Density_model.sgy'write_segy_2d(str(density_output), density_grid, dx=dx, dy=dy)print("\n" + "="*60)print("ПРОВЕРКА ЗАПИСАННЫХ ФАЙЛОВ")print("="*60)# Проверяем записанные файлыfor param_name, output_file in [('Vp', vp_output), ('Vs', vs_output), ('Density', density_output)]:    with segyio.open(str(output_file), 'r', ignore_geometry=True) as f:        data_check = segyio.tools.collect(f.trace[:])        print(f"\n{param_name} (проверка):")        print(f"  Файл: {output_file.name}")        print(f"  Форма: {data_check.shape}")        print(f"  Диапазон: [{data_check.min():.2f}, {data_check.max():.2f}]")        print(f"  Количество трасс: {f.tracecount}")        print(f"  Сэмплов на трассу: {len(f.trace[0])}")print("\n" + "="*60)print("ЗАВЕРШЕНО")print("="*60)

#### Задача 3: Экспорт данных в CSV формат ####

In [ ]:
import csv

In [ ]:
def export_material_to_csv(filename, material_grid, coords_grid, layer_indexes_grid):    """    Экспортирует данные материала в CSV формат с сортировкой по X, затем по Z.    Args:        filename: имя выходного CSV файла        material_grid: массив материала [nx, ny, 3] (E, nu, rho)        coords_grid: массив координат [nx, ny, 2] (x, z)        layer_indexes_grid: массив номеров слоев [nx, ny]    """    nx, ny = material_grid.shape[:2]    # Создаем список всех ячеек    cells_data = []    for i in range(nx):        for j in range(ny):            cell_idx = i * ny + j + 1  # Порядковый индекс начиная с 1            x_coord = coords_grid[i, j, 0]            z_coord = coords_grid[i, j, 1]            layer_num = layer_indexes_grid[i, j]            E_modulus = material_grid[i, j, 0]            poisson_ratio = material_grid[i, j, 1]            density = material_grid[i, j, 2]            cells_data.append([                cell_idx, x_coord, z_coord, layer_num,                E_modulus, poisson_ratio, density            ])    # Сортируем по X, затем по Z    cells_data.sort(key=lambda x: (x[1], x[2]))    # Записываем в CSV    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:        writer = csv.writer(csvfile)        # Заголовки        writer.writerow([            'Порядковый индекс ячейки',            'Координата по X',            'Координата по Z',            'Номер слоя',            'Модуль Юнга',            'Коэффициент Пуассона',            'Плотность'        ])        # Данные        for row in cells_data:            writer.writerow(row)    print(f"Экспортировано {len(cells_data)} ячеек в файл: {filename}")

In [ ]:
# Экспортируем данные в CSVcsv_output = output_dir / 'dev_2_1_material_data.csv'export_material_to_csv(str(csv_output), material_grid, coords_grid, layer_indexes_grid)# Проверяем результатprint(f"\nПроверка экспорта:")print(f"  Файл: {csv_output}")print(f"  Существует: {csv_output.exists()}")if csv_output.exists():    # Читаем первые и последние несколько строк    with open(csv_output, 'r', encoding='utf-8') as f:        lines = f.readlines()        print(f"  Всего строк: {len(lines)}")        print("  Первые 5 строк:")        for i in range(min(6, len(lines))):  # header + 5 строк            print(f"    {lines[i].strip()}")        if len(lines) > 10:            print("  Последние 5 строк:")            for i in range(len(lines)-5, len(lines)):                print(f"    {lines[i].strip()}")

### ВыводыВ Главе I части второй была выполнена подготовка данных для конечно-разностного моделирования в Tesseral:1. **Анализ формата:** Изучена структура файлов SEG-Y из примера Tesseral2. **Конвертация данных:** Преобразованы декартовы данные материала в формат SEG-Y3. **Экспорт в CSV:** Создана таблица со всеми данными материала для дополнительного анализа4. **Выходные файлы:**   - data/dev_2_1_Vp_model.sgy - модель скоростей продольных волн   - data/dev_2_1_Vs_model.sgy - модель скоростей поперечных волн   - data/dev_2_1_Density_model.sgy - модель плотности   - data/dev_2_1_material_data.csv - полная таблица данных материалаДанные готовы для использования в программе Tesseral для конечно-разностного моделирования.